# LIBERO Object data preparation

## Purpose
LIBERO Object data preparation within the MLC Robotics language-conditioned YOLO policy programme.

## Environment and inputs
Internet required; GPU optional. Downloads ten task HDF5 files.

## Execution
The code cells run sequentially using Run All. Embedded installation code provisions the required project modules.

## Outputs
Episode manifest, training-only statistics and data-pipeline checks.

## Evidence and interpretation
450 training and 50 validation episodes in the baseline split. Dataset downloads can occupy several GB.

See docs/notebook_guides/libero_object_data_pipeline.md for the experiment record.


In [ ]:
import sys, subprocess, json, hashlib, re
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','h5py','huggingface_hub','pillow'],check=True)
import numpy as np
import h5py
from PIL import Image
from huggingface_hub import HfApi, snapshot_download
ROOT=Path('/kaggle/working/libero_data_stage')
ROOT.mkdir(exist_ok=True)
REPO='yifengzhu-hf/LIBERO-datasets'
revision=HfApi().dataset_info(REPO).sha
DATA=Path(snapshot_download(REPO,repo_type='dataset',revision=revision,
    allow_patterns=['libero_object/*.hdf5'],local_dir=str(ROOT/'dataset')))
SOURCE=ROOT/'LIBERO'
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Lifelong-Robot-Learning/LIBERO.git',str(SOURCE)],check=True)
source_sha=subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
files=sorted((DATA/'libero_object').glob('*.hdf5'))
assert len(files)==10, f'Expected 10 task files, found {len(files)}'
print('Dataset revision:',revision,'LIBERO revision:',source_sha)


In [ ]:
CAMERAS=('agentview_rgb','eye_in_hand_rgb')
STATE_KEYS=('ee_pos','ee_ori','gripper_states')
HORIZON=16
RESOLUTION=256
SEED=42
manifest=[]
schema={}
for path in files:
    task=path.stem.removesuffix('_demo')
    candidates=list(SOURCE.rglob(task+'.bddl'))
    candidates=[p for p in candidates if 'libero_object' in p.parts]
    assert len(candidates)==1, f'Cannot resolve instruction for {task}: {candidates}'
    content=candidates[0].read_text()
    match=re.search(r'\(:language\s+([^)]*)\)',content,re.S)
    assert match, f'Missing BDDL language: {task}'
    instruction=' '.join(match.group(1).split()).strip('"')
    with h5py.File(path,'r') as f:
        demos=sorted(f['data'].keys())
        assert len(demos)>=2
        for name in demos:
            ep=f['data'][name]; obs=ep['obs']; actions=ep['actions']
            T=len(actions)
            assert T>0 and actions.ndim==2 and actions.shape[1]==7
            schema.setdefault(task,{k:{'shape':list(v.shape),'dtype':str(v.dtype)} for k,v in obs.items()})
            for key in CAMERAS:
                assert key in obs, f'{task}: missing {key}; found {list(obs)}'
                assert obs[key].shape[0]==T and obs[key].shape[-1]==3 and obs[key].dtype==np.uint8
            for key in STATE_KEYS:
                assert key in obs and obs[key].ndim==2 and len(obs[key])==T, f'Invalid state {key}'
            assert sum(obs[k].shape[1] for k in STATE_KEYS)==8
            manifest.append(dict(path=str(path),task=task,episode=name,length=T,instruction=instruction))
# Independent deterministic split within every task, never across frames.
for task in sorted({e['task'] for e in manifest}):
    episodes=[e for e in manifest if e['task']==task]
    rng=np.random.default_rng(SEED+int(hashlib.sha256(task.encode()).hexdigest()[:8],16))
    indices=rng.permutation(len(episodes))
    held=set(indices[:max(1,round(.1*len(episodes)))].tolist())
    for i,e in enumerate(episodes):e['split']='validation' if i in held else 'train'
train=[e for e in manifest if e['split']=='train']
valid=[e for e in manifest if e['split']=='validation']
assert {(e['task'],e['episode']) for e in train}.isdisjoint({(e['task'],e['episode']) for e in valid})
print('Episodes:',len(train),'train,',len(valid),'validation')
print(json.dumps(schema[manifest[0]['task']],indent=2))


In [ ]:
# Fit z-score statistics only on training episodes, each real timestep once.
def fit_stats(episodes):
    sums={}; squares={}; count=0
    for e in episodes:
        assert e['split']=='train', 'Validation leakage into normalisation'
        with h5py.File(e['path'],'r') as f:
            d=f['data'][e['episode']]
            values={'actions':np.asarray(d['actions'],dtype=np.float64),
                    'state':np.concatenate([d['obs'][k][:] for k in STATE_KEYS],axis=1).astype(np.float64)}
            for key,x in values.items():
                assert np.isfinite(x).all()
                sums[key]=sums.get(key,0)+x.sum(0)
                squares[key]=squares.get(key,0)+(x*x).sum(0)
            count+=e['length']
    return {k:{'mean':(sums[k]/count).tolist(),
               'std':np.maximum(np.sqrt(np.maximum(squares[k]/count-(sums[k]/count)**2,0)),1e-6).tolist()}
            for k in sums}
stats=fit_stats(train)
def normalise(x,key):
    return ((x-np.asarray(stats[key]['mean']))/np.asarray(stats[key]['std'])).astype(np.float32)
def denormalise(x,key):
    return x*np.asarray(stats[key]['std'])+np.asarray(stats[key]['mean'])

def sample(e,t,horizon=HORIZON):
    if not 0<=t<e['length'] or horizon<1:raise ValueError('Invalid timestep or horizon')
    with h5py.File(e['path'],'r') as f:
        d=f['data'][e['episode']]; obs=d['obs']
        images=np.stack([np.asarray(Image.fromarray(obs[k][t]).resize((RESOLUTION,RESOLUTION),Image.Resampling.BILINEAR))
                         for k in CAMERAS]).transpose(0,3,1,2).astype(np.float32)/255
        state=np.concatenate([obs[k][t] for k in STATE_KEYS]).astype(np.float32)
        raw=np.asarray(d['actions'][t:min(t+horizon,e['length'])],dtype=np.float32)
    n=len(raw)
    # Zero padding after normalisation; mask MUST be applied to training losses.
    actions=np.zeros((horizon,7),dtype=np.float32)
    actions[:n]=normalise(raw,'actions')
    mask=np.arange(horizon)<n
    return dict(images=images,state=normalise(state,'state'),actions=actions,
                action_mask=mask,instruction=e['instruction'],task=e['task'],episode=e['episode'],t=t)

class EpisodeSamples:
    # Opens HDF5 per access, so workers never share inherited HDF5 handles.
    def __init__(self,episodes):
        self.episodes=episodes
        self.ends=np.cumsum([e['length'] for e in episodes])
    def __len__(self):return int(self.ends[-1]) if len(self.ends) else 0
    def __getitem__(self,index):
        if not 0<=index<len(self):raise IndexError(index)
        i=int(np.searchsorted(self.ends,index,side='right'))
        start=int(self.ends[i-1]) if i else 0
        return sample(self.episodes[i],index-start)
training_data=EpisodeSamples(train)
validation_data=EpisodeSamples(valid)


In [ ]:
# Validate every episode's start and tail against the underlying HDF5 arrays.
checked=0
for e in manifest:
    for t in sorted({0,e['length']-1,max(0,e['length']-HORIZON+1)}):
        s=sample(e,t)
        n=min(HORIZON,e['length']-t)
        assert s['images'].shape==(2,3,RESOLUTION,RESOLUTION)
        assert s['state'].shape==(8,) and s['actions'].shape==(HORIZON,7)
        assert s['action_mask'].sum()==n and s['action_mask'][:n].all()
        assert not s['action_mask'][n:].any() and (s['actions'][n:]==0).all()
        assert np.isfinite(s['images']).all() and 0<=s['images'].min()<=s['images'].max()<=1
        with h5py.File(e['path'],'r') as f:
            d=f['data'][e['episode']]
            np.testing.assert_allclose(denormalise(s['actions'][:n],'actions'),d['actions'][t:t+n],atol=1e-5,rtol=1e-5)
            np.testing.assert_allclose(denormalise(s['state'],'state'),np.concatenate([d['obs'][k][t] for k in STATE_KEYS]),atol=1e-5,rtol=1e-5)
        checked+=1
# Dataset index transitions must start at t=0 in the next episode.
for i,end in enumerate(training_data.ends[:-1]):
    assert training_data[int(end)-1]['t']==train[i]['length']-1
    assert training_data[int(end)]['t']==0
    assert training_data[int(end)]['episode']==train[i+1]['episode']
try:fit_stats(valid[:1])
except AssertionError:pass
else:raise AssertionError('Leakage guard failed')
report={'status':'passed','episode_samples_checked':checked,'train_episodes':len(train),
        'validation_episodes':len(valid),'train_timesteps':len(training_data),
        'dataset_revision':revision,'libero_revision':source_sha,
        'cameras':CAMERAS,'state_keys':STATE_KEYS,'horizon':HORIZON,
        'resolution':RESOLUTION,'observation_history':1,'seed':SEED,
        'image_orientation':'as stored; preview review pending',
        'normalisation':'training-only z-score; all seven action dimensions; no clipping',
        'evaluation_scope':'held-out demonstrations within tasks; not unseen tasks'}
for name,value in [('manifest',manifest),('normalisation',stats),('schema',schema),('validation',report)]:
    (ROOT/(name+'.json')).write_text(json.dumps(value,indent=2))
print(json.dumps(report,indent=2))


In [ ]:
from IPython.display import display
s=training_data[0]
print(s['instruction'], '| camera order:', CAMERAS)
preview=Image.new('RGB',(RESOLUTION*2,RESOLUTION))
for i,img in enumerate(s['images']):
    preview.paste(Image.fromarray((img.transpose(1,2,0)*255).round().astype(np.uint8)),(i*RESOLUTION,0))
preview.save(ROOT/'camera_preview.png')
display(preview)
# Export the implementation and evidence so this stage can be integrated later.
import zipfile
with zipfile.ZipFile('/kaggle/working/libero_data_validation.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.glob('*.json'):z.write(p,p.name)
    z.write(ROOT/'camera_preview.png','camera_preview.png')
print('Return this executed notebook, including its camera preview, and libero_data_validation.zip.')
